In [23]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import os 
import rasterio
from rasterio.transform import rowcol
from rasterio.transform import xy
from rasterio.windows import Window

# Data Explanation : 
1. Fauna : all animals distribution (amphibians , birds , mammals , reptiles) the data gives th nombre of birds , mammals ...etc in this area for example each pixel containe 1km²and the value is he number of animals in this area (form 0 --> 153  or 0 --> 178)

2. vegetation : the percentage of a specific land cover  (barren , evergreen, herbaceous) in speicific area (pixel)

3. human activities : gives the human modification in 1km² area between 0 and 1 

we can deduct somethings like : 
- High Species Richness + Low Human Modification → Biodiversity hotspots with low disturbance (ideal for conservation).
- High Species Richness + High Human Modification → Threatened ecosystems (urgent conservation priority).
- Low Species Richness + High Human Modification → Urban/agricultural areas with low ecological value.
- Low Species Richness + Low Human Modification → Natural areas with fewer species (e.g., deserts).

In [54]:
df = pd.read_csv("CombinedData.csv")
df = df.drop(columns=["USDA Soil Taxonomy" , "altitude"])
df = df.dropna(subset=["X" , "Y"])
df = df[(df['X'] >= 0) & (df['X'] <= 20) & ((df['Y'] >= 20) & (df['Y'] <= 50))]
df.fillna(df.mean(), inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2440 entries, 0 to 2891
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          2440 non-null   int64  
 1   X                   2440 non-null   float64
 2   Y                   2440 non-null   float64
 3   UD                  2440 non-null   float64
 4   LD                  2440 non-null   float64
 5   average profondeur  2440 non-null   float64
 6   pH                  2440 non-null   float64
 7   CE                  2440 non-null   float64
 8   CEC                 2440 non-null   float64
 9   A%                  2440 non-null   float64
 10  L%                  2440 non-null   float64
 11  S%                  2440 non-null   float64
 12  ALR(L)              2440 non-null   float64
 13  ALR(S)              2440 non-null   float64
 14  OM                  2440 non-null   float64
dtypes: float64(14), int64(1)
memory usage: 305.0 KB


In [55]:
df = df.drop(columns=["Unnamed: 0"])
df

,X,Y,UD,LD,average profondeur,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM
0,3.10700,36.684000,15.0,30.0,22.5,7.870,0.170,17.929000,46.172091,45.613249,8.214660,-0.005289,-0.749790,2.019000
1,3.10700,36.684000,30.0,60.0,45.0,7.774,0.170,17.207000,40.361496,37.281777,22.356727,-0.034471,-0.256559,1.503000
2,3.10700,36.684000,60.0,100.0,80.0,7.799,0.170,15.049000,49.608530,42.317043,8.074427,-0.069041,-0.788445,0.706000
3,3.13333,36.071667,15.0,30.0,22.5,7.950,0.260,40.610953,32.330400,40.460456,27.209144,0.097420,-0.074896,2.645000
4,3.13333,36.071667,30.0,60.0,45.0,8.096,0.279,40.610953,33.898341,39.248386,26.853273,0.063643,-0.101181,1.097000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2887,2.43000,27.190000,0.0,8.0,4.0,7.390,2.860,27.887979,24.101929,35.531762,39.888796,4.599834,0.139389,2.975432
2888,2.43000,27.190000,8.0,34.0,21.0,7.650,4.870,27.887979,24.101929,35.531762,39.888796,4.599834,0.139389,0.530000
2889,2.43000,27.190000,34.0,43.0,38.5,7.990,5.480,27.887979,24.101929,35.531762,39.888796,4.599834,0.139389,1.060000
2890,2.43000,27.190000,43.0,60.0,51.5,8.040,4.070,27.887979,24.101929,35.531762,39.888796,4.599834,0.139389,0.000000


In [63]:
df_unique = df.copy()
df_unique = df_unique.drop_duplicates(subset=["X" ,"Y"])
df_unique 

,X,Y,UD,LD,average profondeur,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM
0,3.107000,36.684000,15.0,30.0,22.5,7.87,0.170000,17.929000,46.172091,45.613249,8.214660,-0.005289,-0.749790,2.019000
3,3.133330,36.071667,15.0,30.0,22.5,7.95,0.260000,40.610953,32.330400,40.460456,27.209144,0.097420,-0.074896,2.645000
5,3.150000,36.724167,15.0,30.0,22.5,8.50,2.632808,40.610953,13.000000,32.000000,55.000000,0.391206,0.626419,7.270000
7,2.880278,36.757500,5.0,15.0,10.0,8.59,0.370000,40.610953,7.210000,11.570000,81.220000,0.205397,1.051727,4.100000
9,3.178361,36.704083,15.0,30.0,22.5,7.83,2.632808,3.750000,27.239966,42.857252,29.902781,4.599834,0.139389,1.360000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2870,2.410000,27.180000,0.0,23.0,11.5,8.43,20.000000,6.720000,0.460000,35.531762,39.888796,4.599834,0.139389,0.800000
2873,2.250000,27.350000,0.0,20.0,10.0,7.63,1.150000,27.887979,24.101929,35.531762,39.888796,4.599834,0.139389,2.380000
2879,2.410000,27.960000,0.0,10.0,5.0,6.72,3.710000,27.887979,24.101929,35.531762,39.888796,4.599834,0.139389,1.060000
2884,2.450000,27.200000,0.0,3.0,1.5,6.80,3.750000,27.887979,24.101929,35.531762,39.888796,4.599834,0.139389,1.320000


In [76]:
def latlon_to_pixel(lat, lon, dataset ,transform):
    # Validate if the coordinate is within the image bounds
    if not (dataset.bounds.left <= lon <= dataset.bounds.right and
            dataset.bounds.bottom <= lat <= dataset.bounds.top):
        print("error here ")

    # Convert to pixel row & column
    row, col = rowcol(dataset.transform, lon, lat)
    return row, col

In [65]:
# windows Forming  for full one file (nefthou el file mera brk)
def window_forming(raster_path, df, height=7, width=7 , var=None):
    covariable_files = os.listdir(raster_path)
    covariable_num = len(covariable_files)

    datasets = [rasterio.open(os.path.join(raster_path, fname)) for fname in covariable_files]
    transforms = [ds.transform for ds in datasets]

    windows = []
    mid = height // 2

    for i in range(len(df)):
        print(f"Processing point {i}")
        lat = df.iloc[i]["X"]
        lon = df.iloc[i]["Y"]
        window = np.empty((height, width, covariable_num))

        for idx, src in enumerate(datasets):
            transform = transforms[idx]
            if var :
                pixel_y, pixel_x = latlon_to_pixel(lon, lat, src, transform)
            else : 
                pixel_y, pixel_x = latlon_to_pixel(lat, lon, src, transform)

            try:
                win = Window(pixel_x - mid, pixel_y - mid, width, height)
                data = src.read(1, window=win)
                window[:, :, idx] = data

            except Exception as e:
                print(f"Error at point {i}, covariable {idx}")
                print(e)
                window[:, :, idx] = np.nan  # Fill with NaNs on failure

        windows.append(window)

    for ds in datasets:
        ds.close()

    return windows

In [66]:
# for devided tiff files 
def window_forming2(raster_path  , df ,height=7 , width=7 ):
    covariable_num = len(os.listdir(raster_path))
    windows = []

    for i in range(len(df)) :

        print(f"the dataset is in : {i}") 
        lat =  df.iloc[i]["X"]
        lon = df.iloc[i]["Y"]

        window = np.empty((height, width, covariable_num))
        for index , j in enumerate(os.listdir(raster_path)):
            with rasterio.open(f"{raster_path}/{j}") as src :
                raster_data = src.read()
                transform = src.transform 
                
            try : 
                pixel_x ,pixel_y = latlon_to_pixel(lat , lon , src , transform)
                mid = height//2
                window[:,:,index] = raster_data[ 0 , pixel_x - mid : pixel_x + mid +1 , pixel_y - mid : pixel_y + mid +1 ]
                break 
            except Exception as e :
                continue 
        
        windows.append(window)
    return windows

In [67]:
def window_forming3(all_files  , df ,height=7 , width=7 ):
    covariable_num = len(all_files)
    windows = []
    for i in range(len(df)) :
        print(f"the dataset is in : {i}") 
        lat =  df.iloc[i]["X"]
        lon = df.iloc[i]["Y"]

        window = np.empty((height, width, covariable_num))

        for index , i in enumerate(all_files):
            raster_data = i.raster_data
            pixel_x ,pixel_y = latlon_to_pixel(lat , lon ,  i.src , i.transform)
            print(pixel_x , pixel_y)
            mid = height//2
            try : 
                window[:,:,index] = raster_data[ pixel_x - mid : pixel_x + mid +1 , pixel_y - mid : pixel_y + mid +1 ]
            except Exception as e:
                print("error is here ")
                print(e)
        windows.append(window)
    return windows

### 1.Climat

In [68]:
climat_window = window_forming("../Resmapled/climat/" , df_unique)
climat_window = np.array(climat_window)
np.save("../Data_multi_CNN/climat.npy", climat_window)

Processing point 0
Processing point 1
Processing point 2
Processing point 3
Processing point 4
Processing point 5
Processing point 6
Processing point 7
Processing point 8
Processing point 9
Processing point 10
Processing point 11
Processing point 12
Processing point 13
Processing point 14
Processing point 15
Processing point 16
Processing point 17
Processing point 18
Processing point 19
Processing point 20
Processing point 21
Processing point 22
Processing point 23
Processing point 24
Processing point 25
Processing point 26
Processing point 27
Processing point 28
Processing point 29
Processing point 30
Processing point 31
Processing point 32
Processing point 33
Processing point 34
Processing point 35
Processing point 36
Processing point 37
Processing point 38
Processing point 39
Processing point 40
Processing point 41
Processing point 42
Processing point 43
Processing point 44
Processing point 45
Processing point 46
Processing point 47
Processing point 48
Processing point 49
Processing

In [69]:
climat_window.shape

(448, 7, 7, 6)

### 2. organisme

In [81]:
organisme = window_forming("../Resmapled/organisme/" , df_unique)
organisme = np.array(organisme)
print(organisme.shape)
np.save("../Data_multi_CNN/organisme.npy", organisme)

Processing point 0
Processing point 1
Processing point 2
Processing point 3
Processing point 4
Processing point 5
Processing point 6
Processing point 7
Processing point 8
Processing point 9
Processing point 10
Processing point 11
Processing point 12
Processing point 13
Processing point 14
Processing point 15
Processing point 16
Processing point 17
Processing point 18
Processing point 19
Processing point 20
Processing point 21
Processing point 22
Processing point 23
Processing point 24
Processing point 25
Processing point 26
Processing point 27
Processing point 28
Processing point 29
Processing point 30
Processing point 31
Processing point 32
Processing point 33
Processing point 34
Processing point 35
Processing point 36
Processing point 37
Processing point 38
Processing point 39
Processing point 40
Processing point 41
Processing point 42
Processing point 43
Processing point 44
Processing point 45
Processing point 46
Processing point 47
Processing point 48
Processing point 49
Processing

### 3. relief

In [71]:
relief = window_forming("../Resmapled/relief/" , df_unique)
relief = np.array(relief)
print(relief.shape)
np.save("../Data_multi_CNN/relief.npy", relief)

Processing point 0
Processing point 1
Processing point 2
Processing point 3
Processing point 4
Processing point 5
Processing point 6
Processing point 7
Processing point 8
Processing point 9
Processing point 10
Processing point 11
Processing point 12
Processing point 13
Processing point 14
Processing point 15
Processing point 16
Processing point 17
Processing point 18
Processing point 19
Processing point 20
Processing point 21
Processing point 22
Processing point 23
Processing point 24
Processing point 25
Processing point 26
Processing point 27
Processing point 28
Processing point 29
Processing point 30
Processing point 31
Processing point 32
Processing point 33
Processing point 34
Processing point 35
Processing point 36
Processing point 37
Processing point 38
Processing point 39
Processing point 40
Processing point 41
Processing point 42
Processing point 43
Processing point 44
Processing point 45
Processing point 46
Processing point 47
Processing point 48
Processing point 49
Processing

### 4. Spatial Position

In [72]:
landuse_window = window_forming2("../Resmapled/saptialPosition/landuse/" , df_unique)
water_window = window_forming2("../Resmapled/saptialPosition/water/" , df_unique)
np.save("../Data_multi_CNN/landuse.npy", landuse_window)
np.save("../Data_multi_CNN/water.npy", water_window)

the dataset is in : 0
the dataset is in : 1
the dataset is in : 2
the dataset is in : 3
the dataset is in : 4
the dataset is in : 5
the dataset is in : 6
the dataset is in : 7
the dataset is in : 8
the dataset is in : 9
the dataset is in : 10
the dataset is in : 11
the dataset is in : 12
the dataset is in : 13
the dataset is in : 14
the dataset is in : 15
the dataset is in : 16
the dataset is in : 17
the dataset is in : 18
the dataset is in : 19
the dataset is in : 20
the dataset is in : 21
the dataset is in : 22
the dataset is in : 23
the dataset is in : 24
the dataset is in : 25
the dataset is in : 26
the dataset is in : 27
the dataset is in : 28
the dataset is in : 29
the dataset is in : 30
the dataset is in : 31
the dataset is in : 32
the dataset is in : 33
the dataset is in : 34
the dataset is in : 35
the dataset is in : 36
the dataset is in : 37
the dataset is in : 38
the dataset is in : 39
the dataset is in : 40
the dataset is in : 41
the dataset is in : 42
the dataset is in : 4

### 4. Parent Material

In [77]:
parent_window = window_forming("../Resmapled/ParentMaterial/", df_unique , var=True)
np.save("../Data_multi_CNN/Parent_Material.npy", parent_window)
np.array(parent_window).shape

Processing point 0
Processing point 1
Processing point 2
Processing point 3
Processing point 4
Processing point 5
Processing point 6
Processing point 7
Processing point 8
Processing point 9
Processing point 10
Processing point 11
Processing point 12
Processing point 13
Processing point 14
Processing point 15
Processing point 16
Processing point 17
Processing point 18
Processing point 19
Processing point 20
Processing point 21
Processing point 22
Processing point 23
Processing point 24
Processing point 25
Processing point 26
Processing point 27
Processing point 28
Processing point 29
Processing point 30
Processing point 31
Processing point 32
Processing point 33
Processing point 34
Processing point 35
Processing point 36
Processing point 37
Processing point 38
Processing point 39
Processing point 40
Processing point 41
Processing point 42
Processing point 43
Processing point 44
Processing point 45
Processing point 46
Processing point 47
Processing point 48
Processing point 49
Processing

(448, 7, 7, 2)

In [ ]:
# shape file transformation
import geopandas as gpd
from rasterio.transform import from_bounds
from rasterio.features import rasterize

def transform_shape(shapefile_path , attribute , output_path):
    gdf = gpd.read_file(shapefile_path)
    gdf["category_numeric"], category_mapping = gdf[attribute].factorize()
    bounds = gdf.total_bounds  
    print("Bounds:", bounds)
    resolution = 0.008983  # -- 1KM
    width = int((bounds[2] - bounds[0]) / resolution)
    height = int((bounds[3] - bounds[1]) / resolution)

    print(f"Raster size: Width={width}, Height={height}")

    transform = from_bounds(*bounds, width, height)
    raster = rasterize(
        [(geom, value) for geom, value in zip(gdf.geometry, gdf["category_numeric"])],
        out_shape=(height, width),
        transform=transform,
        fill=-1, 
        dtype=rasterio.int32
    )

    tiff_path = output_path
    with rasterio.open(
        tiff_path, "w", 
        driver="GTiff",
        height=height, 
        width=width, 
        count=1,
        dtype=rasterio.uint8,
        crs=gdf.crs,  # Use same CRS as shapefile
        transform=transform
    ) as dst:
        dst.write(raster, 1)
    print(f"Saved 1 km raster to {tiff_path}")

In [ ]:
transform_shape("../Maps/ParentMaterial/Geomorphology/Algeria/Algeria_HG.shp" , "AlgGLG"  , "output_1km.tif")
geoMorpho_window = window_forming2("../Maps/ParentMaterial/Geomorphology/tiff" , df_unique)
np.save("../Data_multi_CNN/geoMorpho.npy", geoMorpho_window)
geoMorpho_window

In [78]:
class optimizerKhefoufi:
    def __init__(self , raster_data , transforme , src ):
        self.raster_data = raster_data
        self.transform = transforme
        self.src = src

def data_saver(path):
    files = []
    lat_min = 0 
    lat_max = 10

    lon_min = 30
    lon_max = 40


    for i in os.listdir(path):
        print(i)
        with rasterio.open(f"{path}/{i}") as src :
            raster_data = src.read()
            transform = src.transform 

            pixel_minx , pixel_miny = latlon_to_pixel(lat_min , lon_min ,src, transform)
            pixel_maxx , pixel_maxy = latlon_to_pixel(lat_max , lon_max , src , transform)
            print(pixel_minx , pixel_maxx , pixel_miny , pixel_maxy)
            
            raster_data = raster_data[0 , pixel_minx : pixel_maxx , pixel_miny : pixel_maxy]

            tiff_val = optimizerKhefoufi(raster_data , transform , src)
        files.append(tiff_val)
    return files
all_files = data_saver("../Maps/organisme/Microbial/SoilResp_HeterotrophicResp_1928/data")

FileNotFoundError: [WinError 3] The system cannot find the path specified: '../Maps/organisme/Microbial/SoilResp_HeterotrophicResp_1928/data'

In [ ]:
Microbial_window = window_forming3(all_files , df_unique)
np.save("../Data_multi_CNN/microbial.npy", Microbial_window)
Microbial_window

In [79]:
def pixel_to_latlon(row, col, dataset):
    lon, lat = xy(dataset.transform, row, col)
    return lat, lon

In [80]:
windows = []
raster_data_used = "../Resmapled/organisme/evergreen.tif"
with rasterio.open(raster_data_used) as src: 
    raster = src.read()
    transform = src.transform

    for index in range(len(df_unique)):
        lat = df_unique.iloc[index]["X"]
        lon = df_unique.iloc[index]["Y"]

        pixel_row, pixel_col = latlon_to_pixel(lat, lon, src, transform)
        window = np.zeros((7, 7, 2))  # Only initialize once

        for i in range(pixel_row - 3, pixel_row + 4):  # rows (Y)
            for j in range(pixel_col - 3, pixel_col + 4):  # cols (X)
                lat_val, lon_val = pixel_to_latlon(i, j, src)

                row_idx = i - (pixel_row - 3)
                col_idx = j - (pixel_col - 3)

                window[row_idx, col_idx, 0] = lat_val
                window[row_idx, col_idx, 1] = lon_val

        windows.append(window)

# Check result
print(windows[0].shape)
np.save("../Data_multi_CNN/latlon.npy", windows)

(7, 7, 2)


In [ ]:
transform_shape("../Maps/ParentMaterial/Rock_type/geo7_2ag.shp" , "GLG"  , "../Maps/ParentMaterial/Rock_type/tiff/rocksType_1km.tif")
rocks_window = window_forming("../Maps/ParentMaterial/Rock_type/tiff", df_unique)
np.save("../Data_multi_CNN/rocks.npy", rocks_window)
rocks_window

,AREA,PERIMETER,GEO2_7G_,GEO2_7G_ID,GLG,geometry,category_numeric
0,2250.44404,966.65344,2,11939,SEA,"POLYGON ((5.40701 5.57146, 5.4109 5.4694, 5.42...",0
1,1.60529,7.58497,3,6,oth,"POLYGON ((15.64759 40.00266, 15.66 40, 15.676 ...",1
2,7.02790,33.05255,4,3,oth,"POLYGON ((19.89965 39.96591, 22.81549 39.8342,...",1
3,0.06592,1.66218,5,7,oth,"POLYGON ((19.67681 39.78778, 19.7106 39.7867, ...",1
4,0.90818,4.50544,6,4,oth,"POLYGON ((8.40944 39.76892, 9.63446 39.84441, ...",1
...,...,...,...,...,...,...,...
11972,0.01677,0.69987,11974,11935,SO,"POLYGON ((19.69988 -34.58595, 19.70456 -34.581...",31
11973,0.00456,0.35706,11975,11936,D,"POLYGON ((19.69988 -34.58595, 19.70059 -34.602...",27
11974,0.00239,0.20456,11976,11934,pCm,"POLYGON ((19.43258 -34.60229, 19.40437 -34.627...",11
11975,0.00641,0.35766,11977,11938,SO,"POLYGON ((19.84761 -34.67162, 19.85932 -34.661...",31
